# 2026-10-21_1 MG 基因表达 DotPlot（ABCB11 / TTC39B / TFRC / DIO2 / PITPNC1 / COTL1 / APOE）

**需求与口径**

- 细胞范围：仅 MG（Müller glia）；成年剔除 70years（Old_unpaired，仅 macula、无 peripheral 对照）
- 分组：stage × region（region 由 tissue 映射：fovea/macula → central，peripheral → peripheral）
  - 胚胎 stage：Early = 70/79/87/91days，Mid = 100/103/116days，Late = 137/141/142/162/165days
  - 成年 stage：young = 21/25/30/32years，old = 67/75/86years
- X 轴排序：Early-central; Early-peripheral; Mid-central; Mid-peripheral; Late-central; Late-peripheral（成年页：young-central; young-peripheral; old-central; old-peripheral）；Y 轴为基因（按指定顺序，ABCB11 在顶部）
- 胚胎与成年分页：同一 PDF，页 1 = 胚胎，页 2 = 成年
- 颜色：Seurat DotPlot 默认（lightgrey → blue 渐变）；颜色值 avg.exp.scaled 为**跨胚胎+成年全部 10 个分组共享**的 per-gene z-score（合并两期一次 DotPlot 计算，Seurat 默认截断 [-2.5, 2.5]）；点大小 = 表达细胞百分比（scale_radius [0, 6]）
- **图例一致（2026-10-21 修订）**：色标 limits 固定 [-2.5, 2.5]、大小标度 limits 固定 [0, 100]（pct.exp 为百分比），胚胎页与成年页共用同一组标度参数，**两页图例完全一致、跨期可比**
- 仅输出 PDF，不输出 PNG

**数据源（只读，不修改原始对象）**

- 胚胎：`/project/work/2026-05-25_create_summary/Data/2026-06-05_develop_all_human_harmony_anno.rds`
- 成年：`/project/work/2026-05-25_create_summary/Data/2026-06-05_new_adult_human_harmony_anno.rds`

**输出**

- `Plot/2026-10-21_1_MG_gene_expression_dotplot.pdf`（2 页：胚胎 / 成年，**两页图例一致**）
- `Plot/2026-10-21_1_MG_gene_expression_dotplot_regionblock.pdf`（追加版，2 页：胚胎 / 成年；页内 central 块在前 + peripheral 块在后，块内按 stage 排序；标度与上图一致，两页图例一致；**跨期数值不可直接比较**，分页呈现）
- `Data/2026-10-21_1_MG_gene_percell_expr.rds` 与 `.csv`（每细胞 counts + lognorm 表，供后续小提琴图直接复用）
- `Data/2026-10-21_1_MG_gene_dotplot_summary.csv`（每基因 × 分组的 n_cells / avg_exp / avg_exp_scaled / pct_exp；avg_exp_scaled 为跨两期共享 z-score，与图一致）

**备注**

- avg.exp 为大对象 data 层（NormalizeData log-normalized）经 expm1 还原后的线性尺度均值（Seurat DotPlot 默认算法）；pct.exp 为 data > 0 的细胞比例
- 若某基因在某期所有分组的 avg.exp 完全相同（例如全为 0），页内 z-score 为 NaN，对应点显示为灰色
- 分组数较少时 Seurat 提示 `Scaling data with a low number of groups may produce misleading results`，为默认行为，属预期内
- 轴标题沿用 Seurat DotPlot 默认（`Features` = 基因轴，`Identity` = 分组轴）

In [ ]:
# ---- 1. 环境与参数 ----------------------------------------------------------
suppressPackageStartupMessages({
  library(Seurat)
  library(Matrix)
  library(ggplot2)
})
set.seed(1)

DATE_STR <- "2026-10-21_1"
PROJ_DIR <- "/project/work/2026-10-21_violinplot"
DATA_DIR <- file.path(PROJ_DIR, "Data")
PLOT_DIR <- file.path(PROJ_DIR, "Plot")
for (d in c(DATA_DIR, PLOT_DIR)) dir.create(d, showWarnings = FALSE, recursive = TRUE)

# 原始 Seurat 对象（只读）
EMB_RDS <- "/project/work/2026-05-25_create_summary/Data/2026-06-05_develop_all_human_harmony_anno.rds"
ADU_RDS <- "/project/work/2026-05-25_create_summary/Data/2026-06-05_new_adult_human_harmony_anno.rds"

# 目标基因（按指定顺序，图上自上而下）
GENES <- c("ABCB11", "TTC39B", "TFRC", "DIO2", "PITPNC1", "COTL1", "APOE")

# stage 定义（沿用项目口径；成年剔除 70years = Old_unpaired）
STAGE_MAP_EMB <- c(
  "70days" = "Early",  "79days" = "Early",  "87days" = "Early",  "91days" = "Early",
  "100days" = "Mid",   "103days" = "Mid",   "116days" = "Mid",
  "137days" = "Late",  "141days" = "Late",  "142days" = "Late",
  "162days" = "Late",  "165days" = "Late"
)
STAGE_MAP_ADU <- c(
  "21years" = "young", "25years" = "young", "30years" = "young", "32years" = "young",
  "67years" = "old",   "75years" = "old",   "86years" = "old"
)

# 分组展示顺序：stage-central / stage-peripheral 交替
GROUP_LEVELS_EMB <- c("Early-central", "Early-peripheral", "Mid-central",
                      "Mid-peripheral", "Late-central", "Late-peripheral")
GROUP_LEVELS_ADU <- c("young-central", "young-peripheral", "old-central", "old-peripheral")

stopifnot(file.exists(EMB_RDS), file.exists(ADU_RDS))
cat("基因顺序:", paste(GENES, collapse = " -> "), "\n")
cat("胚胎分组顺序:", paste(GROUP_LEVELS_EMB, collapse = "; "), "\n")
cat("成年分组顺序:", paste(GROUP_LEVELS_ADU, collapse = "; "), "\n")
cat("R", as.character(getRversion()), "| Seurat", as.character(packageVersion("Seurat")),
    "| ggplot2", as.character(packageVersion("ggplot2")), "\n")

In [ ]:
# ---- 2. 提取函数：读入对象 -> 仅 MG -> 抽取 7 基因 counts + lognorm -----------
# 说明：
#   * counts 层为原始计数；data 层为大对象 NormalizeData 的 log-normalized 值（直接沿用，不重新归一化）
#   * region：tissue 属于 fovea/macula -> central，peripheral -> peripheral
#   * 无 stage/region 归属的细胞（成年 70years）剔除并记录
extract_mg_table <- function(rds_path, period, stage_map, group_levels) {
  obj <- readRDS(rds_path)
  stopifnot(all(c("celltype", "tissue", "donor_age", "nCount_RNA") %in% colnames(CONTACT_EMAIL)))
  stopifnot(all(GENES %in% rownames(obj)))
  cat(sprintf("[%s] 对象细胞数 %d | 基因数 %d\n", period, ncol(obj), nrow(obj)))

  meta <- CONTACT_EMAIL
  keep <- which(meta$celltype == "MG")
  cells <- rownames(meta)[keep]
  m <- meta[keep, , drop = FALSE]
  cat(sprintf("[%s] MG 细胞数 %d\n", period, length(cells)))

  tissue_l <- tolower(as.character(m$tissue))
  region <- ifelse(tissue_l %in% c("fovea", "macula", "central"), "central",
                   ifelse(tissue_l == "peripheral", "peripheral", NA_character_))
  stage <- unname(stage_map[as.character(m$donor_age)])

  excl <- is.na(stage) | is.na(region)
  if (any(excl)) {
    cat(sprintf("[%s] 剔除无 stage/region 归属细胞 %d 个（donor_age: %s）\n",
                period, sum(excl),
                paste(sort(unique(as.character(m$donor_age[excl]))), collapse = ",")))
  }
  m <- m[!excl, , drop = FALSE]
  region <- region[!excl]
  stage <- stage[!excl]
  cells <- cells[!excl]

  counts_mat <- GetAssayData(obj, assay = "RNA", layer = "counts")[GENES, cells]
  data_mat   <- GetAssayData(obj, assay = "RNA", layer = "data")[GENES, cells]
  stopifnot(identical(colnames(counts_mat), cells))

  counts_df  <- as.data.frame(t(as.matrix(counts_mat)))
  lognorm_df <- as.data.frame(t(as.matrix(data_mat)))
  colnames(counts_df)  <- paste0(GENES, "_counts")
  colnames(lognorm_df) <- paste0(GENES, "_lognorm")

  out <- data.frame(
    cell       = cells,
    period     = period,
    celltype   = "MG",
    donor_age  = as.character(m$donor_age),
    tissue     = as.character(m$tissue),
    region     = region,
    stage      = stage,
    nCount_RNA = as.numeric(m$nCount_RNA),
    stringsAsFactors = FALSE
  )
  out$stage_region <- factor(paste(out$stage, out$region, sep = "-"), levels = group_levels)
  out <- cbind(out, counts_df, lognorm_df)
  rownames(out) <- out$cell
  stopifnot(!any(is.na(out$stage_region)), !any(is.na(out$nCount_RNA)))

  rm(obj, meta, m, counts_mat, data_mat, counts_df, lognorm_df)
  invisible(gc(verbose = FALSE))
  out
}
cat("extract_mg_table() 定义完成\n")

In [ ]:
# ---- 3. 胚胎数据提取 --------------------------------------------------------
emb <- extract_mg_table(EMB_RDS, "Embryonic", STAGE_MAP_EMB, GROUP_LEVELS_EMB)

cat("\n[Embryonic] stage_region 细胞数：\n")
print(table(emb$stage_region))
cat("\n[Embryonic] donor_age（按发育顺序）× stage_region：\n")
print(table(factor(emb$donor_age, levels = names(STAGE_MAP_EMB)), emb$stage_region))
cat("\n[Embryonic] 各基因检出率（counts > 0 细胞占比）：\n")
print(round(colMeans(emb[paste0(GENES, "_counts")] > 0), 4))

In [ ]:
# ---- 4. 成年数据提取（剔除 70years）-----------------------------------------
adu <- extract_mg_table(ADU_RDS, "Adult", STAGE_MAP_ADU, GROUP_LEVELS_ADU)
stopifnot(!("70years" %in% adu$donor_age))
cat("已确认 70years 已剔除\n")

cat("\n[Adult] stage_region 细胞数：\n")
print(table(adu$stage_region))
cat("\n[Adult] donor_age（按年龄顺序）× stage_region：\n")
print(table(factor(adu$donor_age, levels = names(STAGE_MAP_ADU)), adu$stage_region))
cat("\n[Adult] 各基因检出率（counts > 0 细胞占比）：\n")
print(round(colMeans(adu[paste0(GENES, "_counts")] > 0), 4))

In [ ]:
# ---- 5. 合并两期并保存每细胞表达表 -------------------------------------------
emb$stage_region <- as.character(emb$stage_region)
adu$stage_region <- as.character(adu$stage_region)
percell <- rbind(emb, adu)
percell$period <- factor(percell$period, levels = c("Embryonic", "Adult"))
percell$stage_region <- factor(percell$stage_region,
                               levels = c(GROUP_LEVELS_EMB, GROUP_LEVELS_ADU))

cat("合并后 MG 总细胞数:", nrow(percell), "\n")
print(table(percell$period, percell$stage_region))

percell_rds <- file.path(DATA_DIR, paste0(DATE_STR, "_MG_gene_percell_expr.rds"))
percell_csv <- file.path(DATA_DIR, paste0(DATE_STR, "_MG_gene_percell_expr.csv"))
saveRDS(percell, percell_rds)
write.csv(percell, percell_csv, row.names = FALSE, quote = FALSE)
cat("已保存:", basename(percell_rds), "/", basename(percell_csv), "\n")
cat("列名:", paste(colnames(percell), collapse = ", "), "\n")